# Note segmentation — production consolidation and equal dataset weights

Complete `pitch.ipynb` first. Every method uses the selected pitch artifacts. KernelCPD linear calls the **actual production `Recording.detect_notes()`**. Other ruptures methods apply the same production merge to breakpoints: adjacent segment medians must remain within a 0.5-semitone total spread; merged pitch is recomputed from the original frames. Confirmed silence boundaries are preserved.

All candidates then run production alignment, robust timing refitting and repeat recovery. Rank by dataset-macro **Transcription F1**, and report Onset F1: fixed 50 ms onset / 50 cents pitch / max(50 ms, 20% reference duration) offset tolerances. Bach10 and URMP use their annotated note references and isolated score parts, matching the main benchmark's reference parsing.

The minimum duration is `min(factor × shortest fitted score note, cap)`. The previous unmerged note scores and artifacts cannot be reused: a new signature recomputes segmentation/refinement while retaining pitch caches. Method comparison must finish before tuning its winner; provisional choices are not published to mistake tuning.


In [1]:
from pathlib import Path
import os, sys
# Limit native threads before importing numerical libraries: 8 processes × 1 thread.
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ[name] = '1'
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'app.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from experiments.modules.sweeps import AppendixSweeps as sweep
from experiments.modules.sweeps.scripts import note_method_sweeps as note_sweep
from experiments.modules.sweeps.scripts import note_sweep_reporting as note_report
import importlib
sweep = importlib.reload(sweep)
note_sweep = importlib.reload(note_sweep)
import pandas as pd
from experiments.paths import relocated_path
RUN_SWEEPS = False  # Set True with a NEW OUTPUT directory for new experiments.
WORKERS = 8
OUTPUT = ROOT / 'experiments/results/caches/sweeps/balanced_212_v3'
DATASET_ROOT = ROOT / 'experiments/datasets/coco/cocochorales_tiny'
SPLIT = 'test'  # Locally available partition; NEW development cohort, NOT a test claim.
SEED = 20261116


## Frozen cohort and weighting

Pitch and note tuning use **212 stems: 195 existing CocoChorales + 4 Bach10 + 13 URMP**. Each real dataset contributes one stem randomly sampled within each instrument, with seed 20261116. The manifest freezes the selection; restarting does not resample. All 195 Coco stems remain unchanged. These are development samples, not an untouched test set.

Pool metric counts within each dataset, then take the arithmetic mean of the three dataset metrics (one-third each). This is not a pooled 212-track score. `by_dataset.csv` retains individual dataset scores; count columns in aggregate reports are summed counts and do not reconstruct the dataset-macro F1.

Run notebooks in order: **pitch → note → mistake**. The mistake stage uses only the 195 Coco sources, with the selected pitch/note settings. Production defaults and prior output directories remain unchanged.


In [2]:
# Freeze the sample and validate source files; no detector is run here.
manifest = (sweep.materialize(OUTPUT, root=DATASET_ROOT, split=SPLIT,
                              seed=SEED, workers=WORKERS) if RUN_SWEEPS
            else sweep.read(OUTPUT / 'cohort.json'))
cohort = pd.DataFrame(manifest['items'])
assert cohort.groupby('dataset').size().to_dict() == {'bach10-original': 4, 'coco': 195, 'urmp': 13}
display(pd.crosstab(cohort.instrument, cohort.dataset))
print('Frozen manifest:', OUTPUT / 'cohort.json')


dataset,bach10-original,coco,urmp
instrument,,,
bassoon,1,15,1
cello,0,15,1
clarinet,1,15,1
double bass,0,15,1
flute,0,15,1
horn,0,15,1
oboe,0,15,1
saxophone,1,15,1
trombone,0,15,1


Frozen manifest: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/cohort.json


## 1. Compare six methods

Use all **195 CocoChorales stems (15 per instrument)**, plus all 4 Bach10 and 13 URMP stems: **212 tracks per method**, or **1,272 track–method runs** across six methods. Rank with equal dataset weights. All methods use 40 ms pause, 0.5 factor, 200 ms cap and 0.75 pitch step. The selected method is then tuned on all 212 tracks.

Audio / Compute is pooled source duration divided by segmentation + refinement worker wall time, excluding pitch, scoring and cache I/O.


In [3]:
METHODS = tuple(note_sweep.METHOD_CONFIGS)
method_variants = note_sweep.variants_for(METHODS)
display(note_sweep.parameter_table(method_variants))
method_subset = note_sweep.method_selection_items(manifest['items'])
display(pd.DataFrame(method_subset).groupby(['dataset', 'instrument']).size().rename('Tracks'))
print(f'{len(METHODS)} methods × {len(method_subset)} tracks = {len(METHODS) * len(method_subset)} runs')


,Method,Min Pause (ms),Min Note Factor,Min Note Cap (s),Min Pitch Distance (semitones)
0,PELT (L1),40.0,0.5,0.2,0.75
1,PELT (L2),40.0,0.5,0.2,0.75
2,PELT (RBF),40.0,0.5,0.2,0.75
3,KernelCPD (linear),40.0,0.5,0.2,0.75
4,KernelCPD (Gaussian),40.0,0.5,0.2,0.75
5,KernelCPD (cosine),40.0,0.5,0.2,0.75


dataset          instrument 
bach10-original  bassoon         1
                 clarinet        1
                 saxophone       1
                 violin          1
coco             bassoon        15
                 cello          15
                 clarinet       15
                 double bass    15
                 flute          15
                 horn           15
                 oboe           15
                 saxophone      15
                 trombone       15
                 trumpet        15
                 tuba           15
                 viola          15
                 violin         15
urmp             bassoon         1
                 cello           1
                 clarinet        1
                 double bass     1
                 flute           1
                 horn            1
                 oboe            1
                 saxophone       1
                 trombone        1
                 trumpet         1
                 tuba     

6 methods × 212 tracks = 1272 runs


In [4]:
# Complete or resume all six methods before selecting the parameter-stage method.
if RUN_SWEEPS:
    method_results = note_sweep.run(OUTPUT, stage='methods', methods=METHODS, workers=WORKERS)
    method_results = note_report.add_speed(method_results, manifest)
else:
    directory = OUTPUT / 'note_ruptures/77a6988723321244/methods/3867b3fb166a7079'
    method_results = dict(directory=directory,
        summary=pd.read_csv(directory / 'summary.csv'),
        selected_method=sweep.read(directory / 'best.json')['method'])
display(method_results['summary'])


1272/1272 | urmp__40_Miserere_fl_fl_ob_cl_bn__AuSep_5_bn_40_Miserere | KernelCPD (cosine) | Min Pause (ms)=40, Min Note Factor=0.5, Min Note Cap (s)=0.2, Min Pitch Distance (semitones)=0.75 | Cached

,Method,Onset Precision,Onset Recall,Onset F1,Transcription Precision,Transcription Recall,Transcription F1,fp_offset,tp,fn_offset,fp,tp_offset,fn,Min Pause (ms),Min Note Factor,Min Note Cap (s),Min Pitch Distance (semitones),Audio (s) / Compute (s)
0,KernelCPD (linear),0.914961,0.908702,0.911809,0.884550,0.878623,0.881565,680,6928,695,529,6777,544,40.0,0.5,0.2,0.75,49.702702
1,PELT (L2),0.914961,0.908702,0.911809,0.884550,0.878623,0.881565,680,6928,695,529,6777,544,40.0,0.5,0.2,0.75,1.228740
2,PELT (L1),0.913293,0.909217,0.911239,0.882098,0.878236,0.880151,702,6932,698,544,6774,540,40.0,0.5,0.2,0.75,1.032542
3,KernelCPD (Gaussian),0.907952,0.892550,0.900168,0.872353,0.857733,0.864963,749,6820,836,565,6636,652,40.0,0.5,0.2,0.75,63.924501
4,PELT (RBF),0.907952,0.892550,0.900168,0.872353,0.857733,0.864963,749,6820,836,565,6636,652,40.0,0.5,0.2,0.75,7.744771
5,KernelCPD (cosine),0.612594,0.287568,0.385080,0.491025,0.236328,0.314044,1472,2157,5718,1069,1754,5315,40.0,0.5,0.2,0.75,106.246304


In [5]:
print('Selected method:', method_results['selected_method'])
print('Method reports:', method_results['directory'])


Selected method: kernelcpd-linear
Method reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/note_ruptures/77a6988723321244/methods/3867b3fb166a7079


## 2. Review the completed parameter sweep (read-only)

The linear KernelCPD parameter sweep is already complete on all 212 tracks, including all 195 CocoChorales tracks. This section loads its published report; it does not run detection, change caches, or publish a new winner. Expanding the method comparison does not require repeating parameter tuning. These results retain their original recorded configuration and provenance. If the expanded comparison selects another method, it is reported below without automatically starting a new sweep.


In [6]:
# Resolve the already-published parameter result, independently of the new comparison.
completed_selection = sweep.read(OUTPUT / 'note/selection.json')
completed_best_path = relocated_path(completed_selection['best_path'])
if sweep.file_hash(completed_best_path) != completed_selection['best_sha256']:
    raise ValueError('Published parameter report changed; inspect its saved provenance')
completed_directory = completed_best_path.parent
completed_contract = sweep.read(completed_directory / 'contract.json')
if sweep.digest(completed_contract['source']) != completed_selection['source_sha256']:
    raise ValueError('Published parameter provenance does not match its selection record')
completed_winner = sweep.read(completed_best_path)
print('Completed parameter sweep:', note_sweep.METHOD_LABELS[completed_winner['method']])
print('Tracks:', len(completed_winner['artifacts']))
print('Parameter combinations:', len(completed_contract['variants']))
print('Saved reports:', completed_directory)


Completed parameter sweep: KernelCPD (linear)
Tracks: 212
Parameter combinations: 225
Saved reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/note_ruptures/0f6874a3dc85a217/parameters/74ea73487b5b49f9


In [7]:
# READ ONLY: no call to note_sweep.run(stage='parameters').
parameter_results = dict(
    summary=pd.read_csv(completed_directory / 'summary.csv'),
    best_by_method=pd.read_csv(completed_directory / 'best_by_method.csv'),
    directory=completed_directory,
    selected_method=completed_winner['method'],
)
display(parameter_results['summary'].head(15))
print('Completed winning settings:', completed_winner['params'])
print('Per-dataset reports:', completed_directory / 'by_dataset.csv')
if 'method_results' in globals() and method_results['selected_method'] != completed_winner['method']:
    print('Expanded method comparison selected:', note_sweep.METHOD_LABELS[method_results['selected_method']])
    print('The table above remains the completed linear parameter sweep. No new tuning was launched.')


,Method,Onset Precision,Onset Recall,Onset F1,Transcription Precision,Transcription Recall,Transcription F1,fn,fp_offset,tp,fn_offset,fp,tp_offset,Min Pause (ms),Min Note Factor,Min Note Cap (s),Min Pitch Distance (semitones),Audio (s) / Compute (s)
0,KernelCPD (linear),0.925275,0.912836,0.918983,0.894801,0.882965,0.888814,536,603,6936,674,465,6798,20.0,0.75,0.2,0.60,45.472274
1,KernelCPD (linear),0.924302,0.913294,0.918744,0.893968,0.883494,0.888679,532,610,6940,669,473,6803,10.0,0.75,0.2,0.60,44.745607
2,KernelCPD (linear),0.924421,0.912708,0.918502,0.893982,0.882836,0.888350,537,610,6935,675,472,6797,20.0,0.75,0.2,0.50,44.891962
3,KernelCPD (linear),0.923455,0.913165,0.918263,0.893156,0.883365,0.888216,533,617,6939,670,480,6802,10.0,0.75,0.2,0.50,44.642615
4,KernelCPD (linear),0.922917,0.912521,0.917671,0.892349,0.882464,0.887361,538,623,6934,677,484,6795,20.0,0.75,0.2,0.25,45.047296
5,KernelCPD (linear),0.931240,0.904454,0.917413,0.900402,0.874897,0.887236,602,547,6870,738,411,6734,20.0,0.75,0.2,0.75,46.093442
6,KernelCPD (linear),0.922067,0.912850,0.917422,0.891629,0.882864,0.887211,535,629,6937,673,491,6799,10.0,0.75,0.2,0.25,45.137507
7,KernelCPD (linear),0.930188,0.904911,0.917169,0.899494,0.875426,0.887098,598,554,6874,733,419,6739,10.0,0.75,0.2,0.75,45.098705
8,KernelCPD (linear),0.918767,0.914324,0.916526,0.888433,0.884266,0.886331,524,668,6948,663,529,6809,10.0,0.75,0.1,0.60,44.677444
9,KernelCPD (linear),0.918532,0.914080,0.916289,0.888089,0.883894,0.885975,525,667,6947,665,527,6807,10.0,0.50,0.2,0.75,44.707958


Completed winning settings: {'min_note_length_cap': 0.2, 'min_note_length_factor': 0.75, 'min_silence_duration_ms': 20.0, 'pitch_thresh': 0.6}
Per-dataset reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/note_ruptures/0f6874a3dc85a217/parameters/74ea73487b5b49f9/by_dataset.csv


## Linked winner and resumption

Stages share compatible method/parameter/track checkpoints under a new `note_ruptures/<signature>` directory. `note/best.json` and `note/selection.json` are published only after the completed winning-method parameter sweep covers all 212 tracks. The pointer binds the winner to the current pitch winner, source cohort and runner. Mistake tuning rejects missing or stale winners. Old unmerged reports remain preserved.

The full-cohort method comparison reuses compatible per-track checkpoints from the previous 43-track comparison and parameter sweep, including accelerated PELT RBF results. Previous summary tables and winners remain unchanged in their original directory; a fresh winner is chosen only after all six methods cover all 212 tracks.

Frozen winning pitch artifacts remain usable across the exact audited November 17 production-default promotion. The runner still validates artifact hashes, cohort coverage and the saved pitch contract. Current note code/defaults retain their own fingerprint: historical note caches are not reused if those changed.
